In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import cv2
import torch.nn.functional as F
from PIL import Image
from glob import glob
from tqdm import tqdm
from itertools import combinations
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

In [2]:
train_images_path = "data_small/archive/images_labeled/"

In [3]:
IMAGE_WIDTH = 60
IMAGE_HEIGHT = 160
size = (IMAGE_HEIGHT, IMAGE_WIDTH)

In [4]:
class CustomDataset(Dataset):
    def __init__(self, data, path, transform=None):
        self.data = data
        self.path = path
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img1 = Image.open(train_images_path + self.data["image1"][idx])
        img2 = Image.open(train_images_path + self.data["image2"][idx])
        label = self.data["label"][idx]
        
        # Apply image transformations
        if self.transform is not None:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        
        return img1, img2, label

In [5]:
train_data = pd.read_csv("data_small/pairs.csv")
resize = transform=transforms.Compose([transforms.Resize(size),
                                       transforms.ToTensor()
                                     ])
train_dataset = CustomDataset(train_data, train_images_path, transform=resize)

In [ ]:
def visualize_pair(img1, img2, label):
    fig, axes = plt.subplots(1,2)
    axes[0].imshow(np.transpose(img1.numpy(), (1, 2, 0)))
    axes[1].imshow(np.transpose(img2.numpy(), (1, 2, 0)))
    if label:
        fig.suptitle('Same', y=1)
    else:
        fig.suptitle('Different', y=1)

In [ ]:
img1, img2, label = train_dataset[200]
visualize_pair(img1, img2, label)

In [ ]:
img1, img2, label = train_dataset[5000]
visualize_pair(img1, img2, label)

<img src="attachment:86c61ba5-1ad6-4e3d-9cb5-ea7e3bf26cee.png" width=1000>

In [ ]:
class DNN(nn.Module):
    def __init__(self):
        super(DNN, self).__init__()
    
        self.tied_convolution = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=20, kernel_size=5,stride=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(in_channels=20, out_channels=25, kernel_size=5, stride=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        self.patch = nn.Sequential(
            nn.Conv2d(in_channels=25, out_channels=25, kernel_size=5, stride=5),
            nn.ReLU(inplace=True)
        )
        
        self.across_patch = nn.Sequential(
            nn.Conv2d(in_channels=25, out_channels=25, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.relu = nn.ReLU()
        
        self.fc = nn.Sequential(
            
            nn.Linear(4500, 500),
            nn.ReLU(inplace=True),
            
            nn.Linear(500,2)
        )
        
        self.pad = nn.ZeroPad2d(2)
        
        self.softmax = nn.Softmax()
       
    def get_f(self, f):
        _, _, h, w = f.size()
        f = nn.functional.interpolate(f, mode='nearest', size=(h * 5, w * 5))
        f = torch.squeeze(f)
        return f
    
    def get_g(self, y):
        b, c, h, w = y.size()
        g = torch.zeros((b, c, h * 5, w * 5))
        y = self.pad(y)
        for i in range(h):
            for j in range(w):
                a = i*5
                b = j*5
                g[:,:,a:a+5, b:b+5] = y[:,:,i:i+5,j:j+5]
        return g
         
    def cross_input_neighbourhood_difference(self, y1, y2):
        f = self.get_f(y1)
        g = self.get_g(y2)
        return self.relu(f - g)
        
    def forward(self, img1, img2):
        y1 = torch.tensor(img1).float()
        y2 = torch.tensor(img2).float()
        
        y1 = torch.squeeze(y1)
        y2 = torch.squeeze(y2)
        
        y1 = self.tied_convolution(y1)
        y2 = self.tied_convolution(y2)
        
        y1_2 = self.cross_input_neighbourhood_difference(y1, y2)
        y2_1 = self.cross_input_neighbourhood_difference(y2, y1)
    
        y1 = self.patch(y1_2)
        y1 = self.across_patch(y1)
        y2 = self.patch(y2_1)
        y2 = self.across_patch(y2)
        
        y = torch.hstack((y1, y2))
        
        b = y.shape[0]
        y = y.reshape( (b, -1))
        y = self.fc(y)
        
        y = self.softmax(y)
        
        return y

In [6]:
batch_size=64
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)

In [ ]:
counter = []
loss_history = [] 
def train():
    iteration_number= 0
    
    for epoch in range(epochs):
        for i, data in enumerate(train_dataloader, 0):
            img1, img2 , label = data
            print(label)
            
            optimizer.zero_grad()
            y = model(img1, img2)
            loss_contrastive = criterion(y, label)
            loss_contrastive.backward()
            optimizer.step()
            
            if i % 50 == 0:
                print("Epoch number {}\n Current loss {}\n".format(epoch, loss_contrastive.item()))
                iteration_number += 10
                counter.append(iteration_number)
                loss_history.append(loss_contrastive.item())

In [7]:
#preprocessing and loading the data set
class SiameseDataset(Dataset):
    def __init__(self,training_csv,training_dir,transform=None):
        # used to prepare the labels and images path
        self.train_df=pd.read_csv(training_csv)
        self.train_df = self.train_df.drop(columns=['Unnamed: 0'])
        self.train_df.columns =["image1","image2","label"]
        self.train_dir = training_dir   
        self.transform = transform

    def __getitem__(self,index):
        # getting the image path
        image1_path=os.path.join(self.train_dir,self.train_df.iat[index,0])
        image2_path=os.path.join(self.train_dir,self.train_df.iat[index,1])
        # Loading the image
        img0 = Image.open(image1_path)
        img1 = Image.open(image2_path)
        img0 = img0.convert("L")
        img1 = img1.convert("L")
        # Apply image transformations
        if self.transform is not None:
            img0 = self.transform(img0)
            img1 = self.transform(img1)
        return img0, img1 , th.from_numpy(np.array([int(self.train_df.iat[index,2])],dtype=np.float32))
    def __len__(self):
        return len(self.train_df)

In [8]:
training_csv="data_small/pairs.csv"
training_dir="data_small/archive/images_labeled/"
resize = transform=transforms.Compose([transforms.Resize(size),
                                       transforms.ToTensor()
                                     ])
siamese_dataset = SiameseDataset(training_csv, training_dir, transform=resize)

In [9]:
class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        # Setting up the Sequential of CNN Layers
        self.cnn1 = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=5,stride=1),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(5,alpha=0.0001,beta=0.75,k=2),
            nn.MaxPool2d(3, stride=2),
           
            nn.Conv2d(96, 256, kernel_size=5,stride=1,padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(5,alpha=0.0001,beta=0.75,k=2),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout2d(p=0.3),

            nn.Conv2d(256,384 , kernel_size=3,stride=1,padding=1),
            nn.ReLU(inplace=True),
           
            nn.Conv2d(384,256 , kernel_size=3,stride=1,padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout2d(p=0.3),
        )
        # Defining the fully connected layers
        self.fc1 = nn.Sequential(
            nn.Linear(27648, 500), # fix mismatched shape # nn.Linear(4500, 500),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=0.5),
           
            nn.Linear(500, 128), # fix mismatched shape # nn.Linear(1024, 128),
            nn.ReLU(inplace=True),
           
            nn.Linear(128,2))
       
    def forward_once(self, x):
        # Forward pass
        output = self.cnn1(x)
        output = output.view(output.size()[0], -1)
        output = self.fc1(output)
        return output

    def forward(self, input1, input2):
        # forward pass of input 1
        output1 = self.forward_once(input1)
        # forward pass of input 2
        output2 = self.forward_once(input2)
        return output1, output2

In [10]:
class ContrastiveLoss(torch.nn.Module):
    """
    Contrastive loss function.
    Based on:
    """

    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, x0, x1, y):
        # euclidian distance
        diff = x0 - x1
        dist_sq = torch.sum(torch.pow(diff, 2), 1)
        dist = torch.sqrt(dist_sq)

        mdist = self.margin - dist
        dist = torch.clamp(mdist, min=0.0)
        loss = y * dist_sq + (1 - y) * torch.pow(dist, 2)
        loss = torch.sum(loss) / 2.0 / x0.size()[0]
        return loss


In [11]:
net = SiameseNetwork()#.cuda()
# Decalre Loss Function
criterion = ContrastiveLoss()
# Declare Optimizer
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=0.0005)
#train the model
def train():
    epochs=2 # 100
    loss=[]
    counter=[]
    iteration_number = 0
    for epoch in range(1,epochs):
        for i, data in enumerate(train_dataloader,0):
            img0, img1 , label = data
#             img0, img1 , label = img0.cuda(), img1.cuda() , label.cuda()
            optimizer.zero_grad()
            output1,output2 = net(img0,img1)
            loss_contrastive = criterion(output1,output2,label)
            loss_contrastive.backward()
            optimizer.step()   
        print("Epoch {}\n Current loss {}\n".format(epoch,loss_contrastive.item()))
        iteration_number += 10
        counter.append(iteration_number)
        loss.append(loss_contrastive.item())
#     show_plot(counter, loss)  
    return net
#set the device to cuda
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = train()
torch.save(model.state_dict(), "model.pt")
print("Model Saved Successfully")

/usr/local/lib/python3.10/dist-packages/torch/nn/functional.py:1374: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 3D or 4D inputs).
  warnings.warn(warn_msg)


Epoch 1
 Current loss 0.13779975473880768

Model Saved Successfully
